<a href="https://colab.research.google.com/github/zeyneptass/vector-databases-and-rag-fundamentals/blob/main/anomaly_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
!pip install -q pinecone sentence-transformers pandas numpy tqdm

In [49]:
import warnings
import os
import pandas as pd
import numpy as np
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [50]:
from google.colab import userdata

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
pc = Pinecone(api_key=PINECONE_API_KEY)

print("Pinecone istemcisi hazır.")

Pinecone istemcisi hazır.


In [51]:
# Eğitim verisini oluştur
training_data = """%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12346 for outside:192.168.1.1/80 (%ASA-6-302013: Built inbound TCP connection 12345 for inside:10.0.0.1/443 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built inbound TCP connection 12345 for inside:10.0.0.1/443 (%ASA-6-302014: Teardown TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302015: Built inbound UDP connection 12345 for inside:10.0.0.1/53 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302016: Teardown UDP connection 12345 for inside:10.0.0.1/53 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302017: Built inbound ICMP connection for inside:10.0.0.1 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302018: Built inbound TCP connection for inside:10.0.0.1/22 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 (%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80 ("""

# Dosyaya yaz
with open("training.txt", "w") as f:
    f.write(training_data)

print("Eğitim verisi oluşturuldu: training.txt")
print(f"Toplam satır sayısı: {len(training_data.strip().split(chr(10)))}")

Eğitim verisi oluşturuldu: training.txt
Toplam satır sayısı: 1


In [52]:
# Daha çeşitli log satırları
log_samples = [
    "%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80",
    "%ASA-6-302013: Built inbound TCP connection 12346 for inside:10.0.0.1/443",
    "%ASA-6-302014: Teardown TCP connection 12345 for outside:192.168.1.1/80",
    "%ASA-6-302015: Built inbound UDP connection 12347 for inside:10.0.0.1/53",
    "%ASA-6-302016: Teardown UDP connection 12347 for inside:10.0.0.1/53",
    "%ASA-6-302017: Built inbound ICMP connection for inside:10.0.0.1",
    "%ASA-6-302018: Built inbound TCP connection for inside:10.0.0.1/22",
    "%ASA-6-302013: Built outbound TCP connection 12348 for outside:192.168.1.2/443",
]

# Benzer logları eşleştir (yüksek skor)
training_lines = []
for i, log in enumerate(log_samples):
    for j, other in enumerate(log_samples):
        if i == j:
            score = 1.0  # Aynı log
        elif i % 3 == j % 3:
            score = 0.8  # Benzer tür
        else:
            score = 0.2  # Farklı tür
        training_lines.append(f"{log}^{other}^{score}")

with open("training.txt", "w") as f:
    f.write("\n".join(training_lines))

print(f"Eğitim verisi oluşturuldu: {len(training_lines)} satır")

Eğitim verisi oluşturuldu: 64 satır


In [53]:
sample_logs = [
    "%ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80",
    "%ASA-6-302013: Built inbound TCP connection 12346 for inside:10.0.0.1/443",
    "%ASA-6-302014: Teardown TCP connection 12345 for outside:192.168.1.1/80",
    "%ASA-6-302015: Built inbound UDP connection 12347 for inside:10.0.0.1/53",
    "%ASA-6-302016: Teardown UDP connection 12347 for inside:10.0.0.1/53",
    "%ASA-6-302017: Built inbound ICMP connection for inside:10.0.0.1",
    "%ASA-6-302013: Built outbound TCP connection 12348 for outside:192.168.1.2/443",
    "%ASA-6-302013: Built inbound TCP connection 12349 for inside:10.0.0.2/22",
    "%ASA-6-302014: Teardown TCP connection 12346 for outside:192.168.1.1/443",
    "%ASA-6-302013: Built outbound TCP connection 12350 for outside:192.168.1.3/80",
    "%ASA-6-999999: SEGFAULT DETECTED IN THE MATRIX - CRITICAL SYSTEM FAILURE",  # ANOMALİ!
]

with open("sample.log", "w") as f:
    f.write("\n".join(sample_logs))

print(f"Örnek log dosyası oluşturuldu: {len(sample_logs)} satır")
print("Son satır anomali içeriyor!")

Örnek log dosyası oluşturuldu: 11 satır
Son satır anomali içeriyor!


In [54]:
import torch
from sentence_transformers import models

# BERT tabanlı kelime gömme modeli
word_embedding_model = models.Transformer('bert-base-uncased', max_seq_length=256)

# Pooling katmanı (cümle gömmesi için)
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

# Yoğun katman (256 boyuta indirgeme)
dense_model = models.Dense(
    in_features=pooling_model.get_sentence_embedding_dimension(),
    out_features=256,
    activation_function=torch.nn.Tanh()
)

# Modeli birleştir
model = SentenceTransformer(modules=[word_embedding_model, pooling_model, dense_model])

# Cihazı kontrol et
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model {device} üzerinde çalışıyor.")
print(f"Model çıktı boyutu: {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model cpu üzerinde çalışıyor.
Model çıktı boyutu: 256


In [55]:
train_examples = []

with open("training.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("^")
        if len(parts) != 3:
            continue
        log_a, log_b, score = parts
        train_examples.append(InputExample(
            texts=[log_a, log_b],
            label=float(score)
        ))

print(f"Toplam {len(train_examples)} eğitim örneği yüklendi.")

Toplam 64 eğitim örneği yüklendi.


In [56]:
# Veri yükleyici (batch size 16)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Kayıp fonksiyonu (benzerlik öğrenimi için)
train_loss = losses.CosineSimilarityLoss(model)

print("Veri yükleyici ve kayıp fonksiyonu hazır.")

Veri yükleyici ve kayıp fonksiyonu hazır.


In [57]:
# Modeli eğit
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    show_progress_bar=True
)

print("Model eğitildi!")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Model eğitildi!


In [58]:
# Örnek logları oku
with open("sample.log", "r") as f:
    samples = [line.strip() for line in f if line.strip()]

# Gömmeleri üret
embeddings = model.encode(samples, show_progress_bar=True)

print(f"{len(embeddings)} log satırı gömüldü.")
print(f"Gömme boyutu: {embeddings.shape[1]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

11 log satırı gömüldü.
Gömme boyutu: 256


In [59]:
index_name = "anomaly-detection-demo"

# Varsa eski dizini sil
if index_name in pc.list_indexes().names():
    print(f"'{index_name}' siliniyor...")
    pc.delete_index(index_name)

# Yeni dizin oluştur (256 boyut, cosine metriği)
pc.create_index(
    name=index_name,
    dimension=256,  # Model çıktı boyutu
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

print(f"'{index_name}' oluşturuldu.")

'anomaly-detection-demo' siliniyor...
'anomaly-detection-demo' oluşturuldu.


In [60]:
import time

while not pc.describe_index(index_name).status.ready:
    print("Dizin hazırlanıyor...")
    time.sleep(2)

print("Dizin hazır!")
index = pc.Index(index_name)

Dizin hazır!


In [61]:
vectors_to_upload = []

for i, (log, embedding) in enumerate(zip(samples, embeddings)):
    vectors_to_upload.append({
        "id": f"log_{i}",
        "values": embedding.tolist(),
        "metadata": {"text": log[:500]}  # İlk 500 karakter
    })

# Toplu yükleme
batch_size = 100
for i in tqdm(range(0, len(vectors_to_upload), batch_size)):
    batch = vectors_to_upload[i:i + batch_size]
    index.upsert(vectors=batch)

print("Tüm loglar yüklendi!")

  0%|          | 0/1 [00:00<?, ?it/s]

Tüm loglar yüklendi!


In [62]:
stats = index.describe_index_stats()
print(f"Toplam vektör sayısı: {stats['total_vector_count']}")

Toplam vektör sayısı: 11


In [63]:
def detect_anomaly(good_log_index=0, top_k=10):
    """
    Bilinen iyi bir log satırını alıp Pinecone'da arar.
    En düşük skorlu sonuç anomali olabilir.
    """
    good_log = samples[good_log_index]
    print(f"Referans log: {good_log}\n")

    # İyi logun gömülmesini al
    good_embedding = model.encode(good_log).tolist()

    # Pinecone'da ara
    results = index.query(
        vector=good_embedding,
        top_k=top_k,
        include_metadata=True
    )

    # Sonuçları yazdır
    print("En benzer loglar:")
    print("-" * 80)
    for i, match in enumerate(results['matches']):
        score = match['score']
        text = match['metadata']['text'][:80]
        print(f"{i+1}. Skor: {score:.4f} | {text}")
    print("-" * 80)

    # En düşük skorlu sonuç = anomali
    worst_match = results['matches'][-1]
    print(f"\n🚨 ANOMALİ ŞÜPHESİ:")
    print(f"   Skor: {worst_match['score']:.4f}")
    print(f"   Log: {worst_match['metadata']['text']}")

    return results

# Anomaliyi tespit et
results = detect_anomaly(good_log_index=0, top_k=10)

Referans log: %ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80

En benzer loglar:
--------------------------------------------------------------------------------
1. Skor: 0.9993 | %ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80
2. Skor: 0.9974 | %ASA-6-302013: Built outbound TCP connection 12350 for outside:192.168.1.3/80
3. Skor: 0.9898 | %ASA-6-302013: Built outbound TCP connection 12348 for outside:192.168.1.2/443
4. Skor: 0.9843 | %ASA-6-302014: Teardown TCP connection 12345 for outside:192.168.1.1/80
5. Skor: 0.9822 | %ASA-6-302014: Teardown TCP connection 12346 for outside:192.168.1.1/443
6. Skor: 0.9285 | %ASA-6-302013: Built inbound TCP connection 12349 for inside:10.0.0.2/22
7. Skor: 0.9157 | %ASA-6-302013: Built inbound TCP connection 12346 for inside:10.0.0.1/443
8. Skor: 0.9011 | %ASA-6-302015: Built inbound UDP connection 12347 for inside:10.0.0.1/53
9. Skor: 0.8661 | %ASA-6-302016: Teardown UDP connection 12347 f

In [64]:
# Farklı iyi loglarla dene
for i in [1, 2, 3]:
    print(f"\n{'='*80}")
    print(f"Referans log #{i}")
    print('='*80)
    detect_anomaly(good_log_index=i, top_k=10)


Referans log #1
Referans log: %ASA-6-302013: Built inbound TCP connection 12346 for inside:10.0.0.1/443

En benzer loglar:
--------------------------------------------------------------------------------
1. Skor: 1.0000 | %ASA-6-302013: Built inbound TCP connection 12346 for inside:10.0.0.1/443
2. Skor: 0.9868 | %ASA-6-302013: Built inbound TCP connection 12349 for inside:10.0.0.2/22
3. Skor: 0.9702 | %ASA-6-302015: Built inbound UDP connection 12347 for inside:10.0.0.1/53
4. Skor: 0.9519 | %ASA-6-302016: Teardown UDP connection 12347 for inside:10.0.0.1/53
5. Skor: 0.9257 | %ASA-6-302013: Built outbound TCP connection 12348 for outside:192.168.1.2/443
6. Skor: 0.9206 | %ASA-6-302017: Built inbound ICMP connection for inside:10.0.0.1
7. Skor: 0.9156 | %ASA-6-302013: Built outbound TCP connection 12345 for outside:192.168.1.1/80
8. Skor: 0.9156 | %ASA-6-302013: Built outbound TCP connection 12350 for outside:192.168.1.3/80
9. Skor: 0.9146 | %ASA-6-302014: Teardown TCP connection 12346 